REopt Exports (from bundle)

This notebook loads the exported `reopt_bundle_CY####.pkl` created by the main pipeline,
then writes REopt-ready CSVs:
- Per-device propane input profiles (MMBtu/hr) for the analysis year
- Boiler "fixed controls" propane input (MMBtu/hr) [weather-only, scaled, capped+redistributed]
- Electric boiler input (kW) [weather-only, scaled, capped+redistributed]
- BAU electricity + electric boiler merged profile (kW)
- ASHP boiler replacement electric input (kW) using a performance table
- BAU electricity + ASHP merged profile (kW)

Assumptions:
- Bundle was created by the export cell you wrote (exports_for_reopt/reopt_bundle_CY{ANALYSIS_YEAR}.pkl)
- BAU electricity profile CSV exists in the repo (or you provide path)
- ASHP performance table CSV exists (or you provide path)

In [1]:
# setup/repo
import numpy as np
import pandas as pd
import os
import json
import pickle

!rm -rf PGST
!git clone --quiet https://github.com/mickdeines/PGST/
os.chdir("PGST")

In [2]:
# Where the bundle lives (relative to repo root)
EXPORT_DIR = "."

# If you want to hardcode a year, set e.g. 2024; otherwise we'll auto-detect the newest bundle.
ANALYSIS_YEAR_OVERRIDE = None  # e.g. 2024

# BAU electricity profile (must be Hour + Load (kW))
# If you keep the same naming convention as your main notebook:
#   "(REopt) {ANALYSIS_YEAR} BAU Electricity Load Profile.csv"
BAU_FILE_TEMPLATE = "reopt_exports_out/Electricity total loads (kW)/(REopt) {year} BAU Electricity Load Profile.csv"

# ASHP performance table files
PERF_FILES = {
    "LT-ASHP": "HE (140) Performance Table.csv",
    "HT-ASHP": "LE (170) Performance Table.csv",
}

# ASHP options
ENFORCE_CAPACITY = True
COP_FLOOR = 1.0
COP_CEIL  = 10.0
ALLOW_EXTRAPOLATION = False

# If ENFORCE_CAPACITY=True, you can override rated thermal capacity (kWth).
# If None, we default to boiler rated OUTPUT converted to kWth.
ASHP_RATED_KWTH_OVERRIDE = None

# Output directories for generated REopt CSVs
OUT_BASE = "reopt_exports_out"
OUT_PROPANE = os.path.join(OUT_BASE, "Propane input loads (MMBtu-hr)")
OUT_ELEC_INPUT    = os.path.join(OUT_BASE, "Electricity input loads (kW)")
OUT_ELEC_TOTAL    = os.path.join(OUT_BASE, "Electricity total loads (kW)")

os.makedirs(OUT_PROPANE, exist_ok=True)
os.makedirs(OUT_ELEC_INPUT, exist_ok=True)
os.makedirs(OUT_ELEC_TOTAL, exist_ok=True)

In [3]:
# Load bundle
def _find_bundle(export_dir, year_override=None):
    if year_override is not None:
        p = os.path.join(export_dir, f"reopt_bundle_CY{int(year_override)}.pkl")
        if not os.path.exists(p):
            raise FileNotFoundError(f"Bundle not found: {p}")
        return p

    # auto-pick newest by year in filename
    cands = []
    if os.path.isdir(export_dir):
        for fn in os.listdir(export_dir):
            if fn.startswith("reopt_bundle_CY") and fn.endswith(".pkl"):
                cands.append(os.path.join(export_dir, fn))
    if not cands:
        raise FileNotFoundError(f"No bundles found in {export_dir}. Expected reopt_bundle_CY####.pkl")

    def _year_from_path(p):
        base = os.path.basename(p)
        # reopt_bundle_CY2024.pkl
        y = base.replace("reopt_bundle_CY", "").replace(".pkl", "")
        return int(y)

    cands = sorted(cands, key=_year_from_path)
    return cands[-1]

bundle_path = _find_bundle(EXPORT_DIR, ANALYSIS_YEAR_OVERRIDE)
print("Loading bundle:", bundle_path)

with open(bundle_path, "rb") as f:
    bundle = pickle.load(f)

df = bundle["df"].copy()
HEATING_EQUIPMENT = bundle["HEATING_EQUIPMENT"]
NON_HEATING_EQUIPMENT = bundle["NON_HEATING_EQUIPMENT"]
ANALYSIS_YEAR = int(bundle["ANALYSIS_YEAR"])
ANALYSIS_YEARS = bundle["ANALYSIS_YEARS"]
DESIGN_MIN_F = float(bundle["DESIGN_MIN_F"])
BTU_PER_GALLON = float(bundle["BTU_PER_GALLON"])
year_scales = bundle["year_scales"]
tz = bundle.get("tz", None)

print("ANALYSIS_YEAR:", ANALYSIS_YEAR)
print("df shape:", df.shape)
print("tz:", str(df.index.tz))

Loading bundle: ./reopt_bundle_CY2024.pkl
ANALYSIS_YEAR: 2024
df shape: (26304, 8)
tz: America/Los_Angeles


In [4]:
# Core helpers (recreated here so notebook is standalone)
def compute_duty_cycle(temps_f, balance_point=60, design_min=20):
    """
    0% at or above balance_point
    100% at or below design_min
    linear between
    """
    dc = (balance_point - temps_f) / (balance_point - design_min)
    return pd.Series(dc, index=temps_f.index).clip(lower=0.0, upper=1.0)

def cap_and_redistribute(x, cap):
    """Cap hourly array at cap and redistribute clipped energy into headroom hours."""
    x = np.asarray(x, dtype=float).clip(min=0)
    excess = np.clip(x - cap, 0, None).sum()
    x = np.clip(x, None, cap)
    headroom = np.clip(cap - x, 0, None)
    total_hw = headroom.sum()
    if total_hw > 0 and excess > 0:
        x = x + headroom * (excess / total_hw)
    return x

def interp_1d(x, xp, fp, extrapolate=False):
    x = np.asarray(x, dtype=float)
    xp = np.asarray(xp, dtype=float)
    fp = np.asarray(fp, dtype=float)
    if not extrapolate:
        x = np.clip(x, xp.min(), xp.max())
    return np.interp(x, xp, fp)

In [5]:
# Prepare analysis-year slice (8760 hours)
reopt_df = df[df.index.year == ANALYSIS_YEAR].copy()
reopt_df = reopt_df[~((reopt_df.index.month == 2) & (reopt_df.index.day == 29))]

print("REopt export hours:", len(reopt_df))
if len(reopt_df) != 8760:
    print("WARNING: expected 8760 hours. Check data coverage / leap-day handling.")

hours = np.arange(1, len(reopt_df) + 1)

REopt export hours: 8760


In [6]:
# 1) Export per-device propane input profiles (MMBtu/hr)
# These are direct from the model outputs (already scaled), converted from Btu/hr to MMBtu/hr.
device_exports = [
    ("heat_btu_in_unit_1",            f"(REopt) {ANALYSIS_YEAR} Boiler Input Load (MMBtu).csv"),
    ("heat_btu_in_unit_2",            f"(REopt) {ANALYSIS_YEAR} Gym Furnace Input Load (MMBtu).csv"),
    ("heat_btu_in_unit_3",            f"(REopt) {ANALYSIS_YEAR} CFS Furnaces Input Load (MMBtu).csv"),
    ("nonheat_btu_in_ecc_dhw_heater",  f"(REopt) {ANALYSIS_YEAR} ECC DHW Heater Input Load (MMBtu).csv"),
    ("nonheat_btu_in_ecc_stove",       f"(REopt) {ANALYSIS_YEAR} ECC Stove Input Load (MMBtu).csv"),
    ("nonheat_btu_in_admin_stove",     f"(REopt) {ANALYSIS_YEAR} Kitchen Stove Input Load (MMBtu).csv"),
    ("nonheat_btu_in_elder_stove",     f"(REopt) {ANALYSIS_YEAR} Elder Center Stove Input Load (MMBtu).csv"),
]

for col, fn in device_exports:
    if col not in reopt_df.columns:
        print("SKIP missing:", col)
        continue
    out_path = os.path.join(OUT_PROPANE, fn)
    pd.DataFrame({
        "Hour": hours,
        "Heating Load (MMBtu/hr)": reopt_df[col].values / 1e6
    }).to_csv(out_path, index=False)
    print("Wrote:", out_path)

Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 Boiler Input Load (MMBtu).csv
Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 Gym Furnace Input Load (MMBtu).csv
Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 CFS Furnaces Input Load (MMBtu).csv
Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 ECC DHW Heater Input Load (MMBtu).csv
Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 ECC Stove Input Load (MMBtu).csv
Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 Kitchen Stove Input Load (MMBtu).csv
Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 Elder Center Stove Input Load (MMBtu).csv


In [7]:
# 2) Boiler fixed-controls propane input (MMBtu/hr)
# Weather-only duty cycle, no hysteresis mask, no baseline floor.
# Uses `year_scales[ANALYSIS_YEAR]["heat"]` and caps/redistributes at rated input.
uid = "unit_1"
p = HEATING_EQUIPMENT[uid]

# Occupancy mask (legacy single window)
wins = [(p["occupied_start_hr"], p["occupied_end_hr"])]
is_occupied = pd.Series(False, index=reopt_df.index)
for s, e in wins:
    is_occupied |= (
        reopt_df.index.dayofweek.isin(p["occupied_days"]) &
        (reopt_df.index.hour >= int(s)) &
        (reopt_df.index.hour < int(e))
    )

dc_occupied = compute_duty_cycle(reopt_df["temp_f"], balance_point=60, design_min=DESIGN_MIN_F)
dc_setback  = compute_duty_cycle(reopt_df["temp_f"], balance_point=p["setback_temp_f"], design_min=DESIGN_MIN_F) \
              if float(p["setback_temp_f"]) > float(DESIGN_MIN_F) else pd.Series(0.0, index=reopt_df.index)
dc_weather  = dc_occupied.where(is_occupied, dc_setback)

hs = float(year_scales[int(ANALYSIS_YEAR)]["heat"])
rated_in = float(p["rated_input_btu_hr"])

weather_only_btu_in = rated_in * dc_weather.values * hs
weather_only_btu_in = cap_and_redistribute(weather_only_btu_in, cap=rated_in)

propane_mmbtu = weather_only_btu_in / 1e6
out_path = os.path.join(OUT_PROPANE, f"(REopt) {ANALYSIS_YEAR} Boiler Fixed Controls Input Load (MMBtu).csv")
pd.DataFrame({"Hour": hours, "Heating Load (MMBtu/hr)": propane_mmbtu}).to_csv(out_path, index=False)
print("Wrote:", out_path)
print(f"Annual propane input (fixed controls): {propane_mmbtu.sum():,.2f} MMBtu")
print(f"  = {propane_mmbtu.sum() * 1e6 / BTU_PER_GALLON:,.0f} gallons")

Wrote: reopt_exports_out/Propane input loads (MMBtu-hr)/(REopt) 2024 Boiler Fixed Controls Input Load (MMBtu).csv
Annual propane input (fixed controls): 844.39 MMBtu
  = 9,236 gallons


In [8]:
# 3) BAU electricity
FIFTEEN_MIN_FILE = "15-minute Electricity 2023-2025.csv"
ANALYSIS_YEAR = 2024  # or use the one from your reopt bundle

# load 15-min data (index is datetime because you saved with index=True)
elec_15min = pd.read_csv(FIFTEEN_MIN_FILE, index_col=0, parse_dates=True)

# ensure sorted and unique
elec_15min = elec_15min[~elec_15min.index.duplicated(keep="first")].sort_index()

# pick the demand column (your file has "DEMAND (kW)")
s15 = pd.to_numeric(elec_15min["DEMAND (kW)"], errors="coerce")

# hourly average kW
hourly = s15.resample("1h").mean()

# select calendar year
hourly_y = hourly[f"{ANALYSIS_YEAR}-01-01":f"{ANALYSIS_YEAR}-12-31 23:00:00"]

# drop Feb 29 if present
hourly_y = hourly_y[~((hourly_y.index.month == 2) & (hourly_y.index.day == 29))]

if len(hourly_y) != 8760:
    raise ValueError(f"Expected 8760 hours for CY{ANALYSIS_YEAR}, got {len(hourly_y)}. Check coverage.")

bau = pd.DataFrame({
    "Hour": np.arange(1, 8761),
    "Load (kW)": hourly_y.values,
})
bau["Load (kW)"] = bau["Load (kW)"].round(5)

out_path = os.path.join(OUT_ELEC_TOTAL, f"(REopt) {ANALYSIS_YEAR} BAU Electricity Load Profile.csv")
bau.to_csv(out_path, index=False)
print("Wrote:", out_path)

Wrote: reopt_exports_out/Electricity total loads (kW)/(REopt) 2024 BAU Electricity Load Profile.csv


In [9]:
# 4) Electric boiler input (kW) + merge with BAU electricity
existing_boiler_eff = float(p.get("efficiency", 1.0))
electric_in_kw = (weather_only_btu_in * existing_boiler_eff) / 3412.141633

eboiler_path = os.path.join(OUT_ELEC_INPUT, f"(REopt) {ANALYSIS_YEAR} Electric Boiler Input (kW).csv")
pd.DataFrame({"Hour": hours, "Heating Load (kW)": electric_in_kw}).to_csv(eboiler_path, index=False)
print("Wrote:", eboiler_path)

bau_file = BAU_FILE_TEMPLATE.format(year=ANALYSIS_YEAR)
if not os.path.exists(bau_file):
    raise FileNotFoundError(
        f"BAU file not found: {bau_file}\n"
        f"Either place it in repo root or change BAU_FILE_TEMPLATE / path."
    )

bau = pd.read_csv(bau_file)  # Hour, Load (kW)
ebo = pd.read_csv(eboiler_path)  # Hour, Heating Load (kW)

merged = bau.merge(ebo[["Hour", "Heating Load (kW)"]], on="Hour", how="inner")
merged["Load (kW)"] = merged["Load (kW)"] + merged["Heating Load (kW)"]

out_path = os.path.join(OUT_ELEC_TOTAL, f"(REopt) {ANALYSIS_YEAR} Electricity Load Profile with Electric Boiler (kW).csv")
merged[["Hour", "Load (kW)"]].to_csv(out_path, index=False)
print("Wrote:", out_path, "| hours:", len(merged))

Wrote: reopt_exports_out/Electricity input loads (kW)/(REopt) 2024 Electric Boiler Input (kW).csv
Wrote: reopt_exports_out/Electricity total loads (kW)/(REopt) 2024 Electricity Load Profile with Electric Boiler (kW).csv | hours: 8760


In [10]:
# 5) LT/HT-ASHP boiler replacement electric input (kW) + merge with BAU
# Uses same weather_only_btu_in (delivered heat) and converts to kW using COP(T).
# Optionally enforces capacity using CF(T) * ASHP_RATED_KWTH.

Tcol   = "Outdoor Air Temperature for Heating (F)"
COPcol = "Heating COP (kWt/kWe)"
CFcol  = "Heating CF (kWt_max/kWt_rated)"

# Delivered heat required (kWth) from the same weather-only boiler load
Q_load_kwth = (weather_only_btu_in * existing_boiler_eff) / 3412.141633
Tout = reopt_df["temp_f"].values.astype(float)

# Default ASHP rated kWth = boiler rated OUTPUT converted to kWth
boiler_rated_out_btu_hr = rated_in * existing_boiler_eff
default_ashp_rated_kwth = boiler_rated_out_btu_hr / 3412.141633
ashp_rated_kwth = float(ASHP_RATED_KWTH_OVERRIDE) if ASHP_RATED_KWTH_OVERRIDE is not None else float(default_ashp_rated_kwth)

for tag, PERF_FILE in PERF_FILES.items():
    perf = pd.read_csv(PERF_FILE)

    perf[Tcol]   = pd.to_numeric(perf[Tcol], errors="coerce")
    perf[COPcol] = pd.to_numeric(perf[COPcol], errors="coerce")
    if CFcol in perf.columns:
        perf[CFcol] = pd.to_numeric(perf[CFcol], errors="coerce")

    perf = perf.dropna(subset=[Tcol, COPcol]).sort_values(Tcol)

    if CFcol in perf.columns:
        perf[CFcol] = perf[CFcol].interpolate(limit_direction="both")

    T_table   = perf[Tcol].values.astype(float)
    COP_table = perf[COPcol].values.astype(float)
    CF_table  = perf[CFcol].values.astype(float) if CFcol in perf.columns else None

    Tmin, Tmax = float(T_table.min()), float(T_table.max())

    COP = interp_1d(Tout, T_table, COP_table, extrapolate=ALLOW_EXTRAPOLATION)
    COP = np.clip(COP, COP_FLOOR, COP_CEIL)

    if ENFORCE_CAPACITY and (CF_table is not None):
        CF = interp_1d(Tout, T_table, CF_table, extrapolate=ALLOW_EXTRAPOLATION)
        CF = np.clip(CF, 0.0, np.inf)
        Qmax_kwth = ashp_rated_kwth * CF
        Q_ashp_kwth = np.minimum(Q_load_kwth, Qmax_kwth)
    else:
        Q_ashp_kwth = Q_load_kwth

    ashp_kw = Q_ashp_kwth / COP

    # --- write ASHP input file ---
    ashp_path = os.path.join(OUT_ELEC_INPUT, f"(REopt) {ANALYSIS_YEAR} {tag} Electric Input (kW).csv")
    pd.DataFrame({"Hour": hours, "Heating Load (kW)": ashp_kw}).to_csv(ashp_path, index=False)
    print("Wrote:", ashp_path)
    print(f"  {tag}: Temp clamp range used: {Tmin:.1f}F to {Tmax:.1f}F (ALLOW_EXTRAPOLATION={ALLOW_EXTRAPOLATION})")
    print(f"  {tag}: ASHP_RATED_KWTH used: {ashp_rated_kwth:.2f} kWth (ENFORCE_CAPACITY={ENFORCE_CAPACITY})")
    print(f"  {tag}: Annual ASHP kWh (boiler-only): {ashp_kw.sum():,.0f} kWh")

    # --- merge with BAU and write total profile ---
    ashp = pd.read_csv(ashp_path)
    merged = bau.merge(ashp[["Hour", "Heating Load (kW)"]], on="Hour", how="inner")
    merged["Load (kW)"] = merged["Load (kW)"] + merged["Heating Load (kW)"]

    out_path = os.path.join(OUT_ELEC_TOTAL, f"(REopt) {ANALYSIS_YEAR} Electricity Load Profile with {tag} (kW).csv")
    merged[["Hour", "Load (kW)"]].to_csv(out_path, index=False)
    print("Wrote:", out_path, "| hours:", len(merged))

Wrote: reopt_exports_out/Electricity input loads (kW)/(REopt) 2024 LT-ASHP Electric Input (kW).csv
  LT-ASHP: Temp clamp range used: 10.0F to 80.0F (ALLOW_EXTRAPOLATION=False)
  LT-ASHP: ASHP_RATED_KWTH used: 92.02 kWth (ENFORCE_CAPACITY=True)
  LT-ASHP: Annual ASHP kWh (boiler-only): 87,029 kWh
Wrote: reopt_exports_out/Electricity total loads (kW)/(REopt) 2024 Electricity Load Profile with LT-ASHP (kW).csv | hours: 8760
Wrote: reopt_exports_out/Electricity input loads (kW)/(REopt) 2024 HT-ASHP Electric Input (kW).csv
  HT-ASHP: Temp clamp range used: 30.0F to 80.0F (ALLOW_EXTRAPOLATION=False)
  HT-ASHP: ASHP_RATED_KWTH used: 92.02 kWth (ENFORCE_CAPACITY=True)
  HT-ASHP: Annual ASHP kWh (boiler-only): 122,039 kWh
Wrote: reopt_exports_out/Electricity total loads (kW)/(REopt) 2024 Electricity Load Profile with HT-ASHP (kW).csv | hours: 8760
